# 0823_example_001_baseline

> 이 노트북은 작성 형식을 보여주는 **example**입니다. 실제 데이터가 아닌 합성 데이터를 사용합니다.

실제 실험을 시작할 때 이 파일을 복사한 뒤 `MMDD_작성자_실험번호_설명.ipynb` 형식으로 이름을 변경하세요.

## 1. 설정과 라이브러리

실험 ID와 랜덤 시드는 노트북 상단에서 명시합니다.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

EXPERIMENT_ID = "0823_example_001_baseline"
RANDOM_STATE = 42

## 2. 데이터 준비

Example은 익명 컬럼 형태를 모사한 합성 데이터를 사용합니다. 실제 실험에서는 `data/raw/`의 파일을 읽되 원본을 수정하지 않습니다.

In [ ]:
X_array, y_array = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=RANDOM_STATE,
)

X = pd.DataFrame(X_array, columns=["X001", "X002", "X003", "X004"])
y = pd.Series(y_array, name="Y001")

print("shape:", X.shape)
print("target counts:", y.value_counts().sort_index().to_dict())

## 3. 학습·검증 데이터 분리

통계 기반 전처리를 적용하기 전에 데이터를 분리하여 데이터 누수를 방지합니다.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("train shape:", X_train.shape)
print("valid shape:", X_valid.shape)

## 4. 전처리와 모델 학습

전처리기와 모델을 하나의 Pipeline으로 묶어 학습과 예측에 같은 변환을 적용합니다.

In [ ]:
pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)

pipeline.fit(X_train, y_train)

## 5. 평가

In [ ]:
predictions = pipeline.predict(X_valid)
metrics = {
    "accuracy": accuracy_score(y_valid, predictions),
    "f1": f1_score(y_valid, predictions),
}

pd.Series(metrics, name=EXPERIMENT_ID)

## 6. 모델 저장

모델은 노트북 및 실험 문서와 동일한 stem으로 저장합니다. 모델 바이너리는 Git에 커밋하지 않습니다.

In [ ]:
model_path = Path("../models") / f"{EXPERIMENT_ID}.pkl"
joblib.dump(pipeline, model_path)
print(f"saved: {model_path}")

## 7. 결론 및 다음 단계

- 이 노트북은 실험 작성 형식을 보여주기 위한 example입니다.
- 실제 실험에서는 위 평가 출력의 핵심 지표를 여기에 기록합니다.
- 실제 결과와 다음 실험 제안을 대응하는 `docs/experiments/` 문서에도 반영합니다.